In [12]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
import keras_tuner as kt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Dropout
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.optimizers import Adam

In [3]:
df = pd.read_csv('diabetes.csv')

In [4]:
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72.0,35,169.5,33.6,0.627,50,1
1,1,85,66.0,29,102.5,26.6,0.351,31,0
2,8,183,64.0,32,169.5,23.3,0.672,32,1
3,1,89,66.0,23,94.0,28.1,0.167,21,0
4,0,137,40.0,35,168.0,43.1,2.288,33,1


In [5]:
X = df.drop('Outcome',axis=1)
y = df['Outcome']

In [6]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [7]:
scaler = StandardScaler()

In [8]:
X_trains = scaler.fit_transform(X_train)
X_tests = scaler.transform(X_test)

In [10]:
model = Sequential()
model.add(Dense(32,input_dim=8,activation="relu"))
model.add(Dense(1,activation='sigmoid'))
model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 32)                288       
                                                                 
 dense_1 (Dense)             (None, 1)                 33        
                                                                 
Total params: 321
Trainable params: 321
Non-trainable params: 0
_________________________________________________________________


In [15]:

model.compile(loss='binary_crossentropy',metrics=['accuracy'])
model.fit(X,y,epochs=200,validation_split=0.2)

Epoch 1/200
20/20 [==============================] - 1s 11ms/step - loss: 0.6456 - accuracy: 0.6531 - val_loss: 0.6519 - val_accuracy: 0.6429
Epoch 2/200
20/20 [==============================] - 0s 6ms/step - loss: 0.6456 - accuracy: 0.6531 - val_loss: 0.6519 - val_accuracy: 0.6429
Epoch 3/200
20/20 [==============================] - 0s 6ms/step - loss: 0.6456 - accuracy: 0.6531 - val_loss: 0.6520 - val_accuracy: 0.6429
Epoch 4/200
20/20 [==============================] - 0s 6ms/step - loss: 0.6456 - accuracy: 0.6531 - val_loss: 0.6519 - val_accuracy: 0.6429
Epoch 5/200
20/20 [==============================] - 0s 6ms/step - loss: 0.6456 - accuracy: 0.6531 - val_loss: 0.6519 - val_accuracy: 0.6429
Epoch 6/200
20/20 [==============================] - 0s 6ms/step - loss: 0.6455 - accuracy: 0.6531 - val_loss: 0.6520 - val_accuracy: 0.6429
Epoch 7/200
20/20 [==============================] - 0s 6ms/step - loss: 0.6455 - accuracy: 0.6531 - val_loss: 0.6519 - val_accuracy: 0.6429
Epoch 8/200


In [19]:
def build_model(hp):
    model = Sequential()

    model.add(Dense(32,activation='relu',input_dim=8))
    model.add(Dense(1,activation='sigmoid'))

    optimizer = hp.Choice('optimizer',values=['adam','sgd','rmsprop','adadelta'])

    model.compile(optimizer=optimizer, loss='binary_crossentropy',metrics=['accuracy'])

    return model

In [20]:
tuner = kt.RandomSearch(build_model,objective='val_accuracy',max_trials=5)

In [21]:
tuner.search(X_train,y_train,epochs=5,validation_data=(X_test,y_test))

Trial 4 Complete [00h 00m 01s]
val_accuracy: 0.7727272510528564

Best val_accuracy So Far: 0.8181818127632141
Total elapsed time: 00h 00m 05s


In [22]:
tuner.get_best_hyperparameters()[0].values

{'optimizer': 'sgd'}

In [23]:
model = tuner.get_best_models(num_models=1)[0]

In [24]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 32)                288       
                                                                 
 dense_1 (Dense)             (None, 1)                 33        
                                                                 
Total params: 321
Trainable params: 321
Non-trainable params: 0
_________________________________________________________________


In [26]:
model.fit(X_train,y_train,batch_size=32,epochs=200,initial_epoch=5,validation_data=(X_test,y_test))

Epoch 6/200
20/20 [==============================] - 0s 11ms/step - loss: 0.5308 - accuracy: 0.7687 - val_loss: 0.5863 - val_accuracy: 0.7403
Epoch 7/200
20/20 [==============================] - 0s 5ms/step - loss: 0.5227 - accuracy: 0.7752 - val_loss: 0.7928 - val_accuracy: 0.6429
Epoch 8/200
20/20 [==============================] - 0s 5ms/step - loss: 0.5433 - accuracy: 0.7671 - val_loss: 0.5181 - val_accuracy: 0.8117
Epoch 9/200
20/20 [==============================] - 0s 5ms/step - loss: 0.5285 - accuracy: 0.7671 - val_loss: 0.7221 - val_accuracy: 0.6883
Epoch 10/200
20/20 [==============================] - 0s 5ms/step - loss: 0.5287 - accuracy: 0.7524 - val_loss: 0.6277 - val_accuracy: 0.7273
Epoch 11/200
20/20 [==============================] - 0s 5ms/step - loss: 0.4943 - accuracy: 0.8013 - val_loss: 0.8006 - val_accuracy: 0.6948
Epoch 12/200
20/20 [==============================] - 0s 5ms/step - loss: 0.4942 - accuracy: 0.7948 - val_loss: 0.6426 - val_accuracy: 0.6818
Epoch 13/

In [27]:
def build_model(hp):
    model = Sequential()
    units = hp.Int('units',min_value=8,max_value=128,step=1)

    model.add(Dense(units=units,activation='relu',input_dim=8))
    model.add(Dense(1,activation='sigmoid'))

    model.compile(optimizer='rmsprop',loss='binary_crossentropy',metrics=['accuracy'])

    return model

In [28]:
tuner = kt.RandomSearch(build_model,objective='val_accuracy',max_trials=5,directory='mydir')

In [30]:
tuner.search(X_train,y_train,epochs=5,validation_data=(X_test,y_test))

Trial 5 Complete [00h 00m 01s]
val_accuracy: 0.6948052048683167

Best val_accuracy So Far: 0.7922077775001526
Total elapsed time: 00h 00m 45s


In [31]:
tuner.get_best_hyperparameters()[0].values

{'units': 53}

In [32]:
model = tuner.get_best_models(num_models=1)[0]

In [34]:
model.fit(X_train,y_train,batch_size=32,epochs=100,initial_epoch=5)

Epoch 6/100
20/20 [==============================] - 0s 4ms/step - loss: 0.6532 - accuracy: 0.7459
Epoch 7/100
20/20 [==============================] - 0s 4ms/step - loss: 0.6000 - accuracy: 0.7573
Epoch 8/100
20/20 [==============================] - 0s 4ms/step - loss: 0.6422 - accuracy: 0.7345
Epoch 9/100
20/20 [==============================] - 0s 3ms/step - loss: 0.5068 - accuracy: 0.7769
Epoch 10/100
20/20 [==============================] - 0s 4ms/step - loss: 0.5758 - accuracy: 0.7655
Epoch 11/100
20/20 [==============================] - 0s 4ms/step - loss: 0.5983 - accuracy: 0.7459
Epoch 12/100
20/20 [==============================] - 0s 4ms/step - loss: 0.5121 - accuracy: 0.7915
Epoch 13/100
20/20 [==============================] - 0s 4ms/step - loss: 0.5892 - accuracy: 0.7541
Epoch 14/100
20/20 [==============================] - 0s 4ms/step - loss: 0.5185 - accuracy: 0.7769
Epoch 15/100
20/20 [==============================] - 0s 3ms/step - loss: 0.5187 - accuracy: 0.7850
Epoc

In [35]:
def build_model(hp):
    model = Sequential()

    model.add(Dense(72,activation='relu',input_dim=8))

    for i in range(hp.Int('num_layers',min_value=1,max_value=10)):
        model.add(Dense(72,activation='relu'))

    model.add(Dense(1,activation='sigmoid'))
    model.compile(optimizer='rmsprop',loss='binary_crossentropy',metrics=['accuracy'])

    return model

In [36]:
tuner = kt.RandomSearch(build_model,objective='val_accuracy',max_trials=3,directory='mydir',project_name='num_layers')

In [37]:
tuner.search(X_train,y_train,epochs=5,validation_data=(X_test,y_test))

Trial 3 Complete [00h 00m 02s]
val_accuracy: 0.798701286315918

Best val_accuracy So Far: 0.8116883039474487
Total elapsed time: 00h 00m 07s


In [38]:
tuner.get_best_hyperparameters()[0].values

{'num_layers': 3}

In [39]:
model = tuner.get_best_models(num_models=1)[0]

In [40]:
model.fit(X_train,y_train,batch_size=32,epochs=200,initial_epoch=5,validation_data=(X_test,y_test))

Epoch 6/200
20/20 [==============================] - 1s 13ms/step - loss: 0.5253 - accuracy: 0.7932 - val_loss: 0.8409 - val_accuracy: 0.5714
Epoch 7/200
20/20 [==============================] - 0s 8ms/step - loss: 0.5740 - accuracy: 0.7410 - val_loss: 0.8972 - val_accuracy: 0.6429
Epoch 8/200
20/20 [==============================] - 0s 7ms/step - loss: 0.5221 - accuracy: 0.7720 - val_loss: 1.0488 - val_accuracy: 0.4351
Epoch 9/200
20/20 [==============================] - 0s 7ms/step - loss: 0.5276 - accuracy: 0.7590 - val_loss: 0.5702 - val_accuracy: 0.7403
Epoch 10/200
20/20 [==============================] - 0s 7ms/step - loss: 0.5018 - accuracy: 0.7932 - val_loss: 0.6226 - val_accuracy: 0.7662
Epoch 11/200
20/20 [==============================] - 0s 7ms/step - loss: 0.5092 - accuracy: 0.7638 - val_loss: 0.5273 - val_accuracy: 0.8312
Epoch 12/200
20/20 [==============================] - 0s 6ms/step - loss: 0.4512 - accuracy: 0.8143 - val_loss: 0.6709 - val_accuracy: 0.6818
Epoch 13/

In [53]:
def build_model(hp):
    model = Sequential()

    counter = 0

    for i in range(hp.Int('num_layers',min_value=1,max_value=10)):
        if counter == 0:
            model.add(Dense(hp.Int('units' + str(i),min_value=8,max_value=128,step=8),activation=hp.Choice('activation' + str(i),values=['relu','tanh','sigmoid']),input_dim=8))
        else:
            model.add(Dense(hp.Int('units'+str(i),min_value=8,max_value=128,step=8),activation=hp.Choice('activation' +str(i),values=['relu','tanh','sigmoid'])))
        counter+=1
    model.add(Dense(1,activation='sigmoid'))
    model.compile(optimizer=hp.Choice('optimizer',values=['rmsprop','adam','sgd','nadam','adadelta']),loss='binary_crossentropy',metrics=['accuracy'])
    return model

In [54]:
tuner = kt.RandomSearch(build_model, objective='val_accuracy',max_trials=3,directory='mydir',project_name='final1')

Reloading Tuner from mydir\final1\tuner0.json


In [55]:
tuner.search(X_train,y_train,epochs=5,validation_data=(X_test,y_test))

Trial 3 Complete [00h 00m 02s]
val_accuracy: 0.7467532753944397

Best val_accuracy So Far: 0.7467532753944397
Total elapsed time: 00h 01m 46s


In [56]:
tuner.get_best_hyperparameters()[0].values

{'num_layers': 7,
 'units0': 112,
 'activation0': 'tanh',
 'optimizer': 'rmsprop',
 'units1': 8,
 'activation1': 'relu',
 'units2': 8,
 'activation2': 'relu',
 'units3': 8,
 'activation3': 'relu',
 'units4': 8,
 'activation4': 'relu',
 'units5': 8,
 'activation5': 'relu',
 'units6': 8,
 'activation6': 'relu'}

In [58]:
model = tuner.get_best_models(num_models=1)[0]

In [60]:
model.fit(X_train,y_train,epochs=200,initial_epoch=5,validation_data=(X_test,y_test))

Epoch 6/200
20/20 [==============================] - 1s 16ms/step - loss: 0.4937 - accuracy: 0.7736 - val_loss: 0.4922 - val_accuracy: 0.7857
Epoch 7/200
20/20 [==============================] - 0s 9ms/step - loss: 0.4845 - accuracy: 0.7932 - val_loss: 0.4839 - val_accuracy: 0.8052
Epoch 8/200
20/20 [==============================] - 0s 8ms/step - loss: 0.4825 - accuracy: 0.7932 - val_loss: 0.4825 - val_accuracy: 0.7922
Epoch 9/200
20/20 [==============================] - 0s 8ms/step - loss: 0.4789 - accuracy: 0.7883 - val_loss: 0.4835 - val_accuracy: 0.8052
Epoch 10/200
20/20 [==============================] - 0s 9ms/step - loss: 0.4776 - accuracy: 0.7883 - val_loss: 0.4746 - val_accuracy: 0.7987
Epoch 11/200
20/20 [==============================] - 0s 11ms/step - loss: 0.4713 - accuracy: 0.7997 - val_loss: 0.4753 - val_accuracy: 0.7987
Epoch 12/200
20/20 [==============================] - 0s 10ms/step - loss: 0.4721 - accuracy: 0.7997 - val_loss: 0.4835 - val_accuracy: 0.7727
Epoch 1